# Conservative 2D regrid — unstructured mesh + save/load

**Conservative regridding** resamples a gridded field while preserving its
area-weighted integral — the right tool for fluxes and intensive
quantities, where bilinear or nearest-neighbor interpolation would bias
the total.

**Unstructured meshes** — ICON triangles, MPAS hexagons, finite-element
models, generic Voronoi tessellations — are the natural geometry for
simulations that need adaptive resolution (denser cells over land, coarser
over open ocean). They have no `(y, x)` index structure: every cell is
just an arbitrary polygon. So they're never 1D-separable, and the fast
`.regrid.conservative` accessor doesn't apply.

`ConservativeRegridder.from_polygons` takes a flat 1D array of shapely
polygons as source and/or target. The same machinery handles
structured→unstructured, unstructured→structured, and unstructured→
unstructured.

**In this notebook.**

1. Build a synthetic Voronoi mesh as a stand-in for a real ICON/MPAS dataset.
2. Regrid a smooth analytic field from a structured lat/lon source onto
   the mesh.
3. **Persist the regridder to disk** — for a fixed source/target pair the
   weight matrix is the same forever, so saving it lets a long-running
   pipeline (or a follow-up notebook) skip the polygon-intersection build
   on restart.

In [ ]:
import tempfile
from pathlib import Path

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection
from scipy.spatial import Voronoi
import shapely

import xarray_regrid  # noqa: F401
from xarray_regrid import ConservativeRegridder, polygons_from_coords

## Build a Voronoi mesh

Real pipelines load a pre-built mesh (UGRID, ICON, MPAS); for a
self-contained demo we synthesize one — jitter a regular grid of generator
points, take the Voronoi tessellation, and clip to the bounding box. The
construction details aren't the point: `from_polygons` only needs a 1D
array of shapely polygons however we get them.

In [ ]:
def voronoi_mesh(n_points, bbox, seed=0):
    rng = np.random.default_rng(seed)
    x0, y0, x1, y1 = bbox
    side = int(np.sqrt(n_points))
    xs, ys = np.linspace(x0, x1, side), np.linspace(y0, y1, side)
    pts = np.column_stack([np.repeat(xs, side), np.tile(ys, side)])
    pts += rng.normal(scale=(x1 - x0) / side * 0.25, size=pts.shape)
    halo = np.array([
        [2*x0 - x1, 2*y0 - y1], [2*x1 - x0, 2*y0 - y1],
        [2*x0 - x1, 2*y1 - y0], [2*x1 - x0, 2*y1 - y0],
    ])
    vor = Voronoi(np.concatenate([pts, halo]))
    clip = shapely.box(x0, y0, x1, y1)
    polys = []
    for i in range(len(pts)):
        r = vor.regions[vor.point_region[i]]
        if not r or -1 in r:
            continue
        p = shapely.intersection(shapely.Polygon(vor.vertices[r]), clip)
        if p.is_empty or p.geom_type != "Polygon":
            continue
        polys.append(p)
    return np.array(polys, dtype=object)

bbox = (-120, -50, 120, 50)
mesh_polys = voronoi_mesh(n_points=400, bbox=bbox)
print(f"{len(mesh_polys)} mesh cells")

## Structured lat/lon source

A smooth `sin(2λ)·cos(3φ)` field on a 1° rectilinear grid — wavy enough
that the regridded mesh values are visually distinct, smooth enough that
no individual mesh cell aliases the pattern.

In [ ]:
lat_s = np.linspace(-50, 50, 100, endpoint=False) + 0.5
lon_s = np.linspace(-120, 120, 240, endpoint=False) + 0.5
Lo, La = np.meshgrid(lon_s, lat_s)
src = xr.DataArray(
    np.sin(np.deg2rad(Lo) * 2) * np.cos(np.deg2rad(La) * 3),
    dims=("latitude", "longitude"),
    coords={"latitude": lat_s, "longitude": lon_s},
)

## Regrid onto the mesh

`from_polygons` takes flat 1D arrays of source and target polygons. Source
polygons come from the 1D grid coords via `polygons_from_coords` (one
rectangle per cell, built from coordinate midpoints); target polygons are
the Voronoi mesh cells. Source data has to be flattened to a single
`src_cell` dimension to match the flat polygon array.

In [ ]:
rgr = ConservativeRegridder.from_polygons(
    source_polygons=polygons_from_coords(lon_s, lat_s),
    target_polygons=mesh_polys,
    source_dim="src_cell",
    target_dim="cell",
)
print(rgr)

src_flat = xr.DataArray(src.values.ravel(), dims=("src_cell",))
mesh_vals = rgr.regrid(src_flat)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
patches = [np.asarray(p.exterior.coords) for p in mesh_polys]
pc = PolyCollection(patches, array=mesh_vals.values, cmap="RdBu_r",
                    edgecolor="0.4", lw=0.3, clim=(-1, 1))
ax.add_collection(pc)
ax.set_xlim(bbox[0], bbox[2]); ax.set_ylim(bbox[1], bbox[3])
ax.set_aspect("equal")
fig.colorbar(pc, ax=ax, shrink=0.8)
ax.set_title(f"regridded onto {len(mesh_polys)}-cell Voronoi mesh")

## Persist the regridder

The weight matrix is purely a function of source and target geometry —
not of the data — so for a fixed source/target pair it never needs to
change. `to_netcdf` writes it (along with shape and version metadata) to
a small NetCDF; `from_netcdf` rebuilds the regridder. A reload-then-apply
gives bit-identical output to the original, so a long-running pipeline
can skip the (expensive) polygon-intersection build on restart.

In [ ]:
path = Path(tempfile.gettempdir()) / "mesh_regridder.nc"
rgr.to_netcdf(path)

with xr.open_dataset(path) as weights:
    for k in ("xarray_regrid_version", "created", "src_shape", "dst_shape"):
        print(f"  {k}: {weights.attrs[k]}")

rgr2 = ConservativeRegridder.from_netcdf(path)
same = np.array_equal(rgr.regrid(src_flat).values, rgr2.regrid(src_flat).values)
print(f"\nreload bit-identical: {same}")